# **Import Library**

In [1]:
import os
import shutil
import pandas as pd
import glob
from sklearn.model_selection import train_test_split

# Path konfigurasi
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"

# Proporsi pembagian data
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1  

# Mapping kelas asli ke kategori besar (Level 1: Root)
ORGANIC_CLASSES = ['Food Organics', 'Vegetation']
INORGANIC_CLASSES = ['Cardboard', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash']

print("Konfigurasi awal berhasil dimuat.")

Konfigurasi awal berhasil dimuat.


# **Cek Jumlah Data**


### Tujuan

Cell ini bertujuan untuk memastikan bahwa proses pembagian dataset telah berhasil dilakukan sesuai dengan rasio yang ditentukan.

### Proses

Program menghitung jumlah gambar pada setiap subset menggunakan fungsi penghitung file, kemudian membandingkannya dengan jumlah data awal.

Persentase setiap subset dihitung menggunakan

$$
P=\frac{N_{subset}}{N_{total}}
\times100\%
$$

dengan:

- \(N_{subset}\) = jumlah data pada subset tertentu
- \(N_{total}\) = jumlah seluruh dataset

### Implementasi

Hasil verifikasi digunakan untuk memastikan tidak terdapat gambar yang hilang ataupun terduplikasi selama proses pembagian dataset.

In [2]:
image_paths = []
labels = []
extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')

for ext in extensions:
    search_path = os.path.join(RAW_DATA_DIR, "**", ext)
    for path in glob.glob(search_path, recursive=True):
        image_paths.append(path)
        labels.append(os.path.basename(os.path.dirname(path)))

df_data = pd.DataFrame({
    'image_path': image_paths,
    'label': labels
})

print(f"Total keseluruhan data: {len(df_data)} gambar.")

Total keseluruhan data: 9482 gambar.


# **Fungsi buat Menyalin File**


### Tujuan

Cell ini digunakan untuk menyalin setiap gambar ke folder **Train**, **Validation**, atau **Test** sesuai hasil pembagian dataset.

### Proses

Program membaca daftar gambar hasil proses splitting, kemudian menyalin file menggunakan library **shutil** ke folder tujuan.

Secara konseptual proses ini dapat dituliskan sebagai

$$
Image_i
\rightarrow
Subset_j
$$

dengan:

- \(Image_i\) = gambar ke-i
- \(Subset_j\) = Train, Validation, atau Test

### Implementasi

Proses penyalinan dilakukan secara otomatis sehingga seluruh dataset tersusun sesuai struktur yang dibutuhkan pada tahap pelatihan model.

In [3]:
def copy_files(df_subset, target_base_dir, level_type):
    for _, row in df_subset.iterrows():
        # Tentukan nama folder berdasarkan tipenya (organic/inorganic atau nama kelas asli)
        folder_name = row['root_label'] if level_type == 'root' else row['label']
        
        # Buat direktori tujuan jika belum ada
        dest_dir = os.path.join(target_base_dir, folder_name)
        os.makedirs(dest_dir, exist_ok=True)
        
        # Tentukan path file tujuan akhir
        file_name = os.path.basename(row['image_path'])
        dest_path = os.path.join(dest_dir, file_name)
        
        # Salin file gambar
        shutil.copy2(row['image_path'], dest_path)

# **Split Dataset**


### Tujuan

Cell ini digunakan untuk membagi dataset menjadi tiga bagian, yaitu **training**, **validation**, dan **testing**. Pembagian ini bertujuan agar model dapat dilatih, divalidasi, dan diuji menggunakan data yang berbeda sehingga evaluasi model menjadi lebih objektif.

### Proses

Dataset dibagi menggunakan metode **random sampling**, sehingga setiap gambar memiliki peluang yang sama untuk masuk ke salah satu subset.

Jika jumlah seluruh data adalah \(N\), maka pembagian dataset dapat dinyatakan sebagai

$$
N = N_{train}+N_{val}+N_{test}
$$

dengan:

- \(N_{train}\) = jumlah data training
- \(N_{val}\) = jumlah data validation
- \(N_{test}\) = jumlah data testing

Apabila digunakan rasio 70:15:15, maka

$$
N_{train}=0.70N
$$

$$
N_{val}=0.15N
$$

$$
N_{test}=0.15N
$$

### Implementasi

Pembagian dataset dilakukan menggunakan fungsi `train_test_split()` dari Scikit-learn sehingga distribusi data dapat dibagi secara acak sesuai rasio yang telah ditentukan.

In [4]:
df_data['root_label'] = df_data['label'].apply(lambda x: 'organic' if x in ORGANIC_CLASSES else 'inorganic')
df_train, df_remain = train_test_split(
    df_data, 
    test_size=(VAL_RATIO + TEST_RATIO), 
    stratify=df_data['label'], 
    random_state=42
)

# Pisahkan sisa data (20%) tadi menjadi Val (50% dari sisa = 10%) dan Test (50% dari sisa = 10%)
df_val, df_test = train_test_split(
    df_remain, 
    test_size=0.5, 
    stratify=df_remain['label'], 
    random_state=42
)

print(f"Hasil Pembagian Data:")
print(f"| Data Train : {len(df_train)} gambar")
print(f"| Data Val   : {len(df_val)} gambar")
print(f"| Data Test  : {len(df_test)} gambar")

print("\nSedang memproses penyalinan data ke direktori baru, mohon tunggu...")

copy_files(df_train, os.path.join(PROCESSED_DATA_DIR, "level_1_root", "train"), level_type='root')
copy_files(df_val, os.path.join(PROCESSED_DATA_DIR, "level_1_root", "val"), level_type='root')
copy_files(df_test, os.path.join(PROCESSED_DATA_DIR, "level_1_root", "test"), level_type='root')

# Cabang Organik
df_train_org = df_train[df_train['root_label'] == 'organic']
df_val_org = df_val[df_val['root_label'] == 'organic']
df_test_org = df_test[df_test['root_label'] == 'organic']

copy_files(df_train_org, os.path.join(PROCESSED_DATA_DIR, "level_2_sub", "organic_branch", "train"), level_type='sub')
copy_files(df_val_org, os.path.join(PROCESSED_DATA_DIR, "level_2_sub", "organic_branch", "val"), level_type='sub')
copy_files(df_test_org, os.path.join(PROCESSED_DATA_DIR, "level_2_sub", "organic_branch", "test"), level_type='sub')

# Cabang Anorganik
df_train_inorg = df_train[df_train['root_label'] == 'inorganic']
df_val_inorg = df_val[df_val['root_label'] == 'inorganic']
df_test_inorg = df_test[df_test['root_label'] == 'inorganic']

copy_files(df_train_inorg, os.path.join(PROCESSED_DATA_DIR, "level_2_sub", "inorganic_branch", "train"), level_type='sub')
copy_files(df_val_inorg, os.path.join(PROCESSED_DATA_DIR, "level_2_sub", "inorganic_branch", "val"), level_type='sub')
copy_files(df_test_inorg, os.path.join(PROCESSED_DATA_DIR, "level_2_sub", "inorganic_branch", "test"), level_type='sub')

print("\nSelesai! Seluruh data sukses displit dan disusun rapi ke folder 'data/processed/'.")

Hasil Pembagian Data:
| Data Train : 7585 gambar
| Data Val   : 948 gambar
| Data Test  : 949 gambar

Sedang memproses penyalinan data ke direktori baru, mohon tunggu...

Selesai! Seluruh data sukses displit dan disusun rapi ke folder 'data/processed/'.


# **Cek Hasil Akhir**


### Tujuan

Cell ini digunakan untuk memverifikasi bahwa seluruh gambar telah berhasil disalin ke folder hasil preprocessing sesuai dengan struktur dataset yang telah ditentukan.

### Proses

Program membuat fungsi `count_files_in_dir()` untuk menghitung jumlah file pada setiap folder hasil preprocessing.

Jumlah file dihitung menggunakan

$$
N=\sum_{i=1}^{k}1
$$

dengan:

- \(N\) = jumlah file.
- \(k\) = jumlah file yang ditemukan pada direktori.

Selanjutnya program menampilkan jumlah gambar pada beberapa folder, seperti:

- Level 1 Root (Train)
- Sub Organic (Train)
- Sub Inorganic (Train)

### Implementasi

Tahap verifikasi dilakukan untuk memastikan bahwa seluruh proses preprocessing berjalan dengan benar, tidak terdapat file yang hilang, serta struktur folder telah sesuai dengan kebutuhan proses pelatihan model.

In [6]:
def count_files_in_dir(path):
    return len([f for f in glob.glob(os.path.join(path, "**", "*"), recursive=True) if os.path.isfile(f)])

print("Pengecekan Folder Hasil Preprocessing:")
print(f"Total file di Level 1 Root (Train): {count_files_in_dir(os.path.join(PROCESSED_DATA_DIR, 'level_1_root', 'train'))}")
print(f"Total file di Sub-Organic (Train) : {count_files_in_dir(os.path.join(PROCESSED_DATA_DIR, 'level_2_sub', 'organic_branch', 'train'))}")
print(f"Total file di Sub-Inorganic (Train): {count_files_in_dir(os.path.join(PROCESSED_DATA_DIR, 'level_2_sub', 'inorganic_branch', 'train'))}")

Pengecekan Folder Hasil Preprocessing:
Total file di Level 1 Root (Train): 4546
Total file di Sub-Organic (Train) : 816
Total file di Sub-Inorganic (Train): 3730
